### Embed PubMed journal articles into Weaviate

PubMed MultiLabel Text Classification Dataset MeSH: https://www.kaggle.com/datasets/owaiskhan9654/pubmed-multilabel-text-classification

In [ ]:
from weaviate.util import generate_uuid5
import json
import pandas as pd

In [59]:
#Read in the pubmed data
df = pd.read_csv("PubMed Multi Label Text Classification Dataset Processed.csv")

In [60]:
import os
import weaviate
from weaviate.auth import Auth

client = weaviate.connect_to_weaviate_cloud(
    cluster_url=os.environ["WEAVIATE_URL"],
    auth_credentials=Auth.api_key(os.environ["WEAVIATE_API_KEY"]),
)

2026-10-05 11:07:51,745 INFO HTTP Request: GET https://xzztqonlqni1m1kjnjovg.c0.us-east-1.aws.weaviate.cloud/v1/meta "HTTP/1.1 200 OK"
2026-10-05 11:07:52,196 INFO HTTP Request: GET https://pypi.org/pypi/weaviate-client/json "HTTP/1.1 200 OK"


In [61]:
df = df[:500]

In [ ]:
from weaviate.classes.config import Configure, Property, DataType
OLLAMA_URL = os.getenv("NGROK_ENDPOINT")  # ngrok tunnel to local Ollama (port 11434)

In [63]:
client.collections.delete("Articles")
client.collections.create(
    name="Articles",
    properties=[
        Property(name="title", data_type=DataType.TEXT),
        Property(name="abstractText", data_type=DataType.TEXT),
    ],
    vector_config=Configure.Vectors.text2vec_ollama(api_endpoint=OLLAMA_URL, model="nomic-embed-text"),
    generative_config=Configure.Generative.ollama(api_endpoint=OLLAMA_URL, model="gpt-oss:20b"),
)

2026-10-05 11:07:52,356 INFO HTTP Request: DELETE https://xzztqonlqni1m1kjnjovg.c0.us-east-1.aws.weaviate.cloud/v1/schema/Articles "HTTP/1.1 200 OK"
2026-10-05 11:07:52,612 INFO HTTP Request: POST https://xzztqonlqni1m1kjnjovg.c0.us-east-1.aws.weaviate.cloud/v1/schema "HTTP/1.1 200 OK"


In [ ]:
import logging
import numpy as np
from weaviate.util import generate_uuid5

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
# Replace infinity values with NaN and then fill NaN values
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.fillna('', inplace=True)
# Convert columns to string type
df['Title'] = df['Title'].astype(str)
df['abstractText'] = df['abstractText'].astype(str)
# Log the data types
logging.info(f"Title column type: {df['Title'].dtype}")
logging.info(f"abstractText column type: {df['abstractText'].dtype}")
articles = client.collections.get("Articles")
with articles.batch.fixed_size(batch_size=50) as batch:
    for index, row in df.iterrows():
        try:
            question_object = {
                "title": row.Title,
                "abstractText": row.abstractText,
            }
            batch.add_object(
                properties=question_object,
                uuid=generate_uuid5(question_object)
            )
        except Exception as e:
            logging.error(f"Error processing row {index}: {e}")

2026-10-05 11:07:52,629 INFO Title column type: str
2026-10-05 11:07:52,630 INFO abstractText column type: str
2026-10-05 11:07:52,688 INFO HTTP Request: GET https://xzztqonlqni1m1kjnjovg.c0.us-east-1.aws.weaviate.cloud/v1/schema/Articles "HTTP/1.1 200 OK"


In [65]:
print(client.collections.get("Articles").aggregate.over_all(total_count=True))

AggregateReturn(properties={}, total_count=500)


In [ ]:
res = client.collections.get("Articles").query.fetch_objects(limit=2)
print(json.dumps([o.properties for o in res.objects], indent=4))

[
    {
        "abstractText": "AIM: The aim of this study was to assess health-related quality of life (HRQOL) following a stationary cycling intervention in children with cerebral palsy (CP).METHOD: This was a phase I multisite randomized controlled trial with single blinding. HRQOL was evaluated using the Pediatric Quality of Life Inventory SF15 (PedsQL; children) and Pediatric Outcomes Data Collection Instrument (PODCI; parent proxy) before and after a 3-month stationary cycling intervention. Sixty-two children (29 male, 33 female; mean age 11y; range 7-18y) with spastic diplegic CP, classified as levels I to III on the Gross Motor Function Classification System, were enrolled. Paired and independent t-tests were used to evaluate within- and between-group differences respectively.RESULTS: Between-group differences, favoring the cycling group, were found for PedsQL emotional functioning (p=0.046) and Parental PODCI treatment expectations scores (p=0.006). Between-group differences 

In [ ]:
response = client.collections.get("Articles").query.near_text(
    query="Mouth Neoplasms",
    limit=10,
)
print(json.dumps([o.properties for o in response.objects], indent=4))

[
    {
        "abstractText": "We encountered two cases of low malignant mucoepidermoid carcinoma with scanty cellular atypism which originated in the parotid or submandibular gland and was characterized by marked fibrosis and eosinophilic infiltration within tumor tissue despite the predominance of the squamous component. Here we report these two cases and provide a review of the literature. We believe that clinically these two tumors with stromal fibrosis and eosinophilic infiltration have a low malignant potential, although histological examination revealed a scanty mucus-producing epithelial component. Therefore, we consider this type of tumor as a new subtype of mucoepidermoid carcinoma. A low-malignant mucoepidermoid carcinoma with stromal fibrosis and eosinophilic infiltration, as described in these two cases, may be misdiagnosed as a highly malignant mucoepidermoid carcinoma or squamous cell carcinoma because of its histologically scanty mucus-producing epithelial component. 

In [ ]:
from weaviate.classes.query import MetadataQuery
coll = client.collections.get("Articles")
seed = coll.query.fetch_objects(limit=1).objects[0]
response = coll.query.near_object(
    near_object=seed.uuid,
    limit=10,
    return_metadata=MetadataQuery(distance=True),
)
# Drop the seed itself (distance 0) so only genuinely similar articles are shown
others = [o for o in response.objects if o.uuid != seed.uuid]
for o in others:
    print(round(o.metadata.distance, 4), o.properties)

0.226 {'abstractText': "OBJECTIVES: To elicit valid quality of life estimates and the highest acceptable treatment risk of different outcomes after stroke. This is a prerequisite for rational medical decision-making, especially when considering treatments like thrombolysis.SUBJECTS: Healthy people, non-stroke medical patients and stroke survivors aged 20-84 years (n = 158)INTERVENTIONS: Subjects were interviewed by a physician using three different methods ('standard gamble', 'time trade-off' and 'direct scaling') supported by an interactive computer program.MAIN OUTCOME MEASURES: We measured utility, a numerical value ranging from 0.00 (death) to 1.00 (perfect health), representing the strength of the patient's preference for an outcome. When using the standard gamble method, risk is also introduced into the measurement.RESULTS: People's preferences for stroke outcomes varied widely, and the estimates were influenced by assessment method. We found that previous stroke, marital status 

In [ ]:
response = client.collections.get("Articles").query.near_text(
    query="Gingival metastasis as first sign of multiorgan dissemination of epithelioid malignant mesothelioma",
    limit=1,
)
from openai import OpenAI
llm = OpenAI(base_url=os.getenv("OLLAMA_BASE_URL"), api_key="ollama")  # local generation, avoids the tunnel timeout
for o in response.objects:
    PROMPT = "Please explain this article " + o.properties["title"] + " like you would to someone without a medical degree."
    out = llm.chat.completions.create(
        model="gpt-oss:20b",
        messages=[{"role": "user", "content": PROMPT}],
        max_tokens=2000,
    )
    print(out.choices[0].message.content)


In [ ]:
#Grouped RAG
response = client.collections.get("Articles").query.near_text(query="Mouth Neoplasms", limit=3)
context = "\n\n".join(o.properties["title"] + ": " + o.properties["abstractText"] for o in response.objects)
from openai import OpenAI
llm = OpenAI(base_url=os.getenv("OLLAMA_BASE_URL"), api_key="ollama")  # local generation, avoids the tunnel timeout
PROMPT = "Summarize the key information here in bullet points. Make it understandable to someone without a medical degree.\n\n" + context
out = llm.chat.completions.create(
    model="gpt-oss:20b",
    messages=[{"role": "user", "content": PROMPT}],
    max_tokens=2000,
)
print(out.choices[0].message.content)


2026-10-05 11:08:50,517 INFO HTTP Request: POST http://localhost:11434/v1/chat/completions "HTTP/1.1 200 OK"


**Key take‑aways for a general reader**

- **A newly recognized tumor of the salivary glands**  
  - *Name:* Sclerosing mucoepidermoid carcinoma with eosinophilia.  
  - *Where it occurs:* Parotid or submandibular gland (the two major salivary glands).  
  - *What it looks like:*  
    - Thick, fibrous tissue (“stroma”) that makes the tumor feel hard.  
    - Lots of white‑blood‑cell type “eosinophils” marching in the tumor.  
    - Only a *few* cells that normally produce mucus; most cells look squamous (flat‑shaped).  
  - *Why it matters:* Because it can be mistaken for a highly aggressive form of cancer, but it actually behaves more gently (low malignant potential).  The authors suggest it should be counted as its own “subtype” of mucoepidermoid carcinoma.

- **A very rare skin spread of a special breast cancer**  
  - *Cancer type:* Sarcomatoid (also called “metaplastic”) breast carcinoma – a highly aggressive tumor that looks like a mix of usual breast cancer cells and “mesenchym

### Turn metadata into a KG

In [ ]:
from rdflib import Graph, RDF, RDFS, Namespace, URIRef, Literal
from rdflib.namespace import SKOS, XSD
import pandas as pd
import urllib.parse
import random
from datetime import datetime, timedelta

# Create a new RDF graph
g = Graph()

# Define namespaces
schema = Namespace('http://schema.org/')
ex = Namespace('http://example.org/')
prefixes = {
    'schema': schema,
    'ex': ex,
    'skos': SKOS,
    'xsd': XSD
}
for p, ns in prefixes.items():
    g.bind(p, ns)

# Define classes and properties
Article = URIRef(ex.Article)
MeSHTerm = URIRef(ex.MeSHTerm)
g.add((Article, RDF.type, RDFS.Class))
g.add((MeSHTerm, RDF.type, RDFS.Class))

title = URIRef(schema.name)
abstract = URIRef(schema.description)
date_published = URIRef(schema.datePublished)
access = URIRef(ex.access)

g.add((title, RDF.type, RDF.Property))
g.add((abstract, RDF.type, RDF.Property))
g.add((date_published, RDF.type, RDF.Property))
g.add((access, RDF.type, RDF.Property))

# Function to clean and parse MeSH terms
def parse_mesh_terms(mesh_list):
    if pd.isna(mesh_list):
        return []
    return [term.strip().replace(' ', '_') for term in mesh_list.strip("[]'").split(',')]

# Function to create a valid URI
def create_valid_uri(base_uri, text):
    if pd.isna(text):
        return None
    sanitized_text = urllib.parse.quote(text.strip().replace(' ', '_').replace('"', '').replace('<', '').replace('>', '').replace("'", "_"))
    return URIRef(f"{base_uri}/{sanitized_text}")

# Function to generate a random date within the last 5 years
def generate_random_date():
    start_date = datetime.now() - timedelta(days=5*365)
    random_days = random.randint(0, 5*365)
    return start_date + timedelta(days=random_days)

# Function to generate a random access value between 1 and 10
def generate_random_access():
    return random.randint(1, 10)

# Load your DataFrame here
# df = pd.read_csv('your_data.csv')

# Loop through each row in the DataFrame and create RDF triples
for index, row in df.iterrows():
    article_uri = create_valid_uri("http://example.org/article", row['Title'])
    if article_uri is None:
        continue
    
    # Add Article instance
    g.add((article_uri, RDF.type, Article))
    g.add((article_uri, title, Literal(row['Title'], datatype=XSD.string)))
    g.add((article_uri, abstract, Literal(row['abstractText'], datatype=XSD.string)))
    
    # Add random datePublished and access
    random_date = generate_random_date()
    random_access = generate_random_access()
    g.add((article_uri, date_published, Literal(random_date.date(), datatype=XSD.date)))
    g.add((article_uri, access, Literal(random_access, datatype=XSD.integer)))
    
    # Add MeSH Terms
    mesh_terms = parse_mesh_terms(row['meshMajor'])
    for term in mesh_terms:
        term_uri = create_valid_uri("http://example.org/mesh", term)
        if term_uri is None:
            continue        
        # Add MeSH Term instance
        g.add((term_uri, RDF.type, MeSHTerm))
        g.add((term_uri, RDFS.label, Literal(term.replace('_', ' '), datatype=XSD.string)))        
        # Link Article to MeSH Term
        g.add((article_uri, schema.about, term_uri))
# Serialize the graph to a file (optional)
g.serialize(destination='ontology.ttl', format='turtle')

<Graph identifier=N8fc1fc37e9f04bf19215b119ec363571 (<class 'rdflib.graph.Graph'>)>

### Semantic search using KG

In [ ]:
from SPARQLWrapper import SPARQLWrapper, JSON

def get_concept_triples_for_term(term):
    sparql = SPARQLWrapper("https://id.nlm.nih.gov/mesh/sparql")
    query = f"""
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX meshv: <http://id.nlm.nih.gov/mesh/vocab#>
    PREFIX mesh: <http://id.nlm.nih.gov/mesh/>

    SELECT ?subject ?p ?pLabel ?o ?oLabel
    FROM <http://id.nlm.nih.gov/mesh>
    WHERE {{
        ?subject rdfs:label "{term}"@en .
        ?subject ?p ?o .
        FILTER(CONTAINS(STR(?p), "concept"))
        OPTIONAL {{ ?p rdfs:label ?pLabel . }}
        OPTIONAL {{ ?o rdfs:label ?oLabel . }}
    }}
    """
    
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)
    results = sparql.query().convert()    
    triples = set()  # Using a set to avoid duplicate entries
    for result in results["results"]["bindings"]:
        obj_label = result.get("oLabel", {}).get("value", "No label")
        triples.add(obj_label)    
    # Add the term itself to the list
    triples.add(term)    
    return list(triples)  # Convert back to a list for easier handling

def get_narrower_concepts_for_term(term):
    sparql = SPARQLWrapper("https://id.nlm.nih.gov/mesh/sparql")
    query = f"""
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX meshv: <http://id.nlm.nih.gov/mesh/vocab#>
    PREFIX mesh: <http://id.nlm.nih.gov/mesh/>

    SELECT ?narrowerConcept ?narrowerConceptLabel
    WHERE {{
        ?broaderConcept rdfs:label "{term}"@en .
        ?narrowerConcept meshv:broaderDescriptor ?broaderConcept .
        ?narrowerConcept rdfs:label ?narrowerConceptLabel .
    }}
    """
    
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)
    results = sparql.query().convert()    
    concepts = set()  # Using a set to avoid duplicate entries
    for result in results["results"]["bindings"]:
        subject_label = result.get("narrowerConceptLabel", {}).get("value", "No label")
        concepts.add(subject_label)    
    return list(concepts)  # Convert back to a list for easier handling

def get_all_narrower_concepts(term, depth=2, current_depth=1):
    # Create a dictionary to store the terms and their narrower concepts
    all_concepts = {}
    # Initial fetch for the primary term
    narrower_concepts = get_narrower_concepts_for_term(term)
    all_concepts[term] = narrower_concepts    
    # If the current depth is less than the desired depth, fetch narrower concepts recursively
    if current_depth < depth:
        for concept in narrower_concepts:
            # Recursive call to fetch narrower concepts for the current concept
            child_concepts = get_all_narrower_concepts(concept, depth, current_depth + 1)
            all_concepts.update(child_concepts)    
    return all_concepts

# Fetch alternative names and narrower concepts
term = "Mouth Neoplasms"
alternative_names = get_concept_triples_for_term(term)
all_concepts = get_all_narrower_concepts(term, depth=4)  # Adjust depth as needed

# Output alternative names
print("Alternative names:", alternative_names)
print()

# Output narrower concepts
for broader, narrower in all_concepts.items():
    print(f"Broader concept: {broader}")
    print(f"Narrower concepts: {narrower}")
    print("---")

Alternative names: ['Cancer of Mouth', 'Mouth Neoplasms']

Broader concept: Mouth Neoplasms
Narrower concepts: ['Lip Neoplasms', 'Salivary Gland Neoplasms', 'Gingival Neoplasms', 'Palatal Neoplasms', 'Leukoplakia, Oral', 'Tongue Neoplasms']
---
Broader concept: Lip Neoplasms
Narrower concepts: []
---
Broader concept: Salivary Gland Neoplasms
Narrower concepts: ['Parotid Neoplasms', 'Sublingual Gland Neoplasms', 'Submandibular Gland Neoplasms']
---
Broader concept: Parotid Neoplasms
Narrower concepts: []
---
Broader concept: Sublingual Gland Neoplasms
Narrower concepts: []
---
Broader concept: Submandibular Gland Neoplasms
Narrower concepts: []
---
Broader concept: Gingival Neoplasms
Narrower concepts: []
---
Broader concept: Palatal Neoplasms
Narrower concepts: []
---
Broader concept: Leukoplakia, Oral
Narrower concepts: ['Leukoplakia, Hairy']
---
Broader concept: Leukoplakia, Hairy
Narrower concepts: []
---
Broader concept: Tongue Neoplasms
Narrower concepts: []
---


In [73]:
def flatten_concepts(concepts_dict):
    flat_list = []

    def recurse_terms(term_dict):
        for term, narrower_terms in term_dict.items():
            flat_list.append(term)
            if narrower_terms:
                recurse_terms(dict.fromkeys(narrower_terms, []))  # Use an empty dict to recurse
    
    recurse_terms(concepts_dict)
    return flat_list

# Flatten the concepts dictionary
flat_list = flatten_concepts(all_concepts)

In [ ]:
#Convert the MeSH terms to URI
def convert_to_mesh_uri(term):
    formatted_term = term.replace(" ", "_").replace(",", "_").replace("-", "_")
    return URIRef(f"http://example.org/mesh/_{formatted_term}_")

# Convert terms to URIs
mesh_terms = [convert_to_mesh_uri(term) for term in flat_list]

In [ ]:
from rdflib import URIRef
query = """
PREFIX schema: <http://schema.org/>
PREFIX ex: <http://example.org/>

SELECT ?article ?title ?abstract ?datePublished ?access ?meshTerm
WHERE {
  ?article a ex:Article ;
           schema:name ?title ;
           schema:description ?abstract ;
           schema:datePublished ?datePublished ;
           ex:access ?access ;
           schema:about ?meshTerm .

  ?meshTerm a ex:MeSHTerm .
}
"""

# Dictionary to store articles and their associated MeSH terms
article_data = {}
# Run the query for each MeSH term
for mesh_term in mesh_terms:
    results = g.query(query, initBindings={'meshTerm': mesh_term})
    # Process results
    for row in results:
        article_uri = row['article']
        if article_uri not in article_data:
            article_data[article_uri] = {
                'title': row['title'],
                'abstract': row['abstract'],
                'datePublished': row['datePublished'],
                'access': row['access'],
                'meshTerms': set()
            }
        # Add the MeSH term to the set for this article
        article_data[article_uri]['meshTerms'].add(str(row['meshTerm']))

# Rank articles by the number of matching MeSH terms
ranked_articles = sorted(
    article_data.items(),
    key=lambda item: len(item[1]['meshTerms']),
    reverse=True
)

# Get the top 3 articles
top_3_articles = ranked_articles[:3]

# Output results
for article_uri, data in top_3_articles:
    print(f"Title: {data['title']}")
    print(f"Abstract: {data['abstract']}")
    print("MeSH Terms:")
    for mesh_term in data['meshTerms']:
        print(f"  - {mesh_term}")
    print()

Title: NDRG1 deficiency is associated with regional metastasis in oral cancer by inducing epithelial-mesenchymal transition.
Abstract: Regional metastasis is the single most important prognostic factor in oral squamous cell carcinoma (OSCC). Abnormal expression of N-myc downstream-regulated genes (NDRGs) has been identified to occur in several tumor types and to predict poor prognosis. In OSCC, the clinical significance of deregulated NDRG expression has not been fully established. In this study, NDRG1 relevance was assessed at gene and protein levels in 100 OSCC patients followed up by at least 10 years. Survival outcome was analyzed using a multivariable analysis. Tumor progression and metastasis was investigated in preclinical model using oral cancer cell lines (HSC3 and SCC25) treated with epidermal growth factor (EGF) and orthotopic mouse model of metastatic murine OSCC (AT84). We identified NDRG1 expression levels to be significantly lower in patients with metastatic tumors compa

### Similarity search using a KG

In [ ]:
from rdflib import Graph, URIRef
from rdflib.namespace import RDF, RDFS, Namespace, SKOS
import urllib.parse

# Define namespaces
schema = Namespace('http://schema.org/')
ex = Namespace('http://example.org/')
rdfs = Namespace('http://www.w3.org/2000/01/rdf-schema#')

# Function to calculate Jaccard similarity and return overlapping terms
def jaccard_similarity(set1, set2):
    intersection = set1.intersection(set2)
    union = set1.union(set2)
    similarity = len(intersection) / len(union) if len(union) != 0 else 0
    return similarity, intersection

# Load the RDF graph
g = Graph()
g.parse('ontology.ttl', format='turtle')

def get_article_uri(title):
    # Convert the title to a URI-safe string
    safe_title = urllib.parse.quote(title.replace(" ", "_"))
    return URIRef(f"http://example.org/article/{safe_title}")

def get_mesh_terms(article_uri):
    query = """
    PREFIX schema: <http://schema.org/>
    PREFIX ex: <http://example.org/>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

    SELECT ?meshTerm
    WHERE {
      ?article schema:about ?meshTerm .
      ?meshTerm a ex:MeSHTerm .
      FILTER (?article = <""" + str(article_uri) + """>)
    }
    """
    results = g.query(query)
    mesh_terms = {str(row['meshTerm']) for row in results}
    return mesh_terms

def find_similar_articles(title):
    article_uri = get_article_uri(title)
    mesh_terms_given_article = get_mesh_terms(article_uri)

    # Query all articles and their MeSH terms
    query = """
    PREFIX schema: <http://schema.org/>
    PREFIX ex: <http://example.org/>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

    SELECT ?article ?meshTerm
    WHERE {
      ?article a ex:Article ;
               schema:about ?meshTerm .
      ?meshTerm a ex:MeSHTerm .
    }
    """
    results = g.query(query)
    mesh_terms_other_articles = {}
    for row in results:
        article = str(row['article'])
        mesh_term = str(row['meshTerm'])
        if article not in mesh_terms_other_articles:
            mesh_terms_other_articles[article] = set()
        mesh_terms_other_articles[article].add(mesh_term)

    # Calculate Jaccard similarity
    similarities = {}
    overlapping_terms = {}
    for article, mesh_terms in mesh_terms_other_articles.items():
        if article != str(article_uri):
            similarity, overlap = jaccard_similarity(mesh_terms_given_article, mesh_terms)
            similarities[article] = similarity
            overlapping_terms[article] = overlap

    # Sort by similarity and get top 5
    top_similar_articles = sorted(similarities.items(), key=lambda x: x[1], reverse=True)[:15]    
    # Print results
    print(f"Top 15 articles similar to '{title}':")
    for article, similarity in top_similar_articles:
        print(f"Article URI: {article}")
        print(f"Jaccard Similarity: {similarity:.4f}")
        print(f"Overlapping MeSH Terms: {overlapping_terms[article]}")
        print()

# Example usage
article_title = "[A case of acute cholestatic hepatitis associated with Orlistat]."  # must be an article in the 500-row subset
find_similar_articles(article_title)


Top 15 articles similar to '[A case of acute cholestatic hepatitis associated with Orlistat].':
Article URI: http://example.org/article/Effect_of_dietary_advice_and_n-3_supplementation_in_newly_diagnosed_MS_patients.
Jaccard Similarity: 0.1739
Overlapping MeSH Terms: {'http://example.org/mesh/_Humans_', 'http://example.org/mesh/_Female_', 'http://example.org/mesh/_Adult_', 'http://example.org/mesh/Acute_Disease_'}

Article URI: http://example.org/article/Binaural_speech_intelligibility_in_noise_for_hearing-impaired_listeners.
Jaccard Similarity: 0.1667
Overlapping MeSH Terms: {'http://example.org/mesh/_Humans_', 'http://example.org/mesh/_Female_', 'http://example.org/mesh/_Adult_'}

Article URI: http://example.org/article/Downregulation_of_miR-224-5p_Promotes_Migration_and_Proliferation_in_Human_Dental_Pulp_Stem_Cells.
Jaccard Similarity: 0.1667
Overlapping MeSH Terms: {'http://example.org/mesh/_Humans_', 'http://example.org/mesh/_Female_', 'http://example.org/mesh/_Adult_'}

Article U

In [ ]:
from rdflib import URIRef

# Function to retrieve title and abstract for a given article URI
def get_article_details(article_uri):
    query = """
    PREFIX schema: <http://schema.org/>
    PREFIX ex: <http://example.org/>

    SELECT ?title ?abstract ?datePublished ?access
    WHERE {{
      <{}> a ex:Article ;
            schema:name ?title ;
            schema:description ?abstract ;
            schema:datePublished ?datePublished ;
            ex:access ?access .
    }}
    """.format(article_uri)

    # Execute the query
    results = g.query(query)
    # Process results
    for row in results:
        title = row['title']
        abstract = row['abstract']
        date_published = row['datePublished']
        access = row['access']
        return title, abstract, date_published, access
    # Return None if no results found
    return None, None, None, None

# Example usage
article_uri = "http://example.org/article/CT_scan_cerebral_hemispheric_asymmetries%3A_predictors_of_recovery_from_aphasia."  # Use the URI as a string
title, abstract, date_published, access = get_article_details(article_uri)

if title and abstract:
    print(f"Title: {title}")
    print(f"Abstract: {abstract}")
    print(f"Date Published: {date_published}")
    print(f"Access: {access}")
else:
    print("No article found with the given URI.")

No article found with the given URI.


### RAG using a KG

In [78]:
# Function to combine titles and abstracts
def combine_abstracts(top_3_articles):
    combined_text = "".join(
        [f"Title: {data['title']} Abstract: {data['abstract']}" for article_uri, data in top_3_articles]
    )
    return combined_text

# Combine abstracts from the top 3 articles
combined_text = combine_abstracts(top_3_articles)
print(combined_text)


Title: NDRG1 deficiency is associated with regional metastasis in oral cancer by inducing epithelial-mesenchymal transition. Abstract: Regional metastasis is the single most important prognostic factor in oral squamous cell carcinoma (OSCC). Abnormal expression of N-myc downstream-regulated genes (NDRGs) has been identified to occur in several tumor types and to predict poor prognosis. In OSCC, the clinical significance of deregulated NDRG expression has not been fully established. In this study, NDRG1 relevance was assessed at gene and protein levels in 100 OSCC patients followed up by at least 10 years. Survival outcome was analyzed using a multivariable analysis. Tumor progression and metastasis was investigated in preclinical model using oral cancer cell lines (HSC3 and SCC25) treated with epidermal growth factor (EGF) and orthotopic mouse model of metastatic murine OSCC (AT84). We identified NDRG1 expression levels to be significantly lower in patients with metastatic tumors compa

In [ ]:
import pandas as pd
from openai import OpenAI
# Local Ollama through its OpenAI-compatible API (no OpenAI key needed)
llm = OpenAI(base_url=os.getenv("OLLAMA_BASE_URL"), api_key="ollama")

In [ ]:
def generate_summary(combined_text):
    response = llm.chat.completions.create(
        model="gpt-oss:20b",
        messages=[{"role": "user", "content": f"Summarize the key information here in bullet points. Make it understandable to someone without a medical degree:\n\n{combined_text}"}],
        max_tokens=1000,
        temperature=0.3,
    )
    
    # Get the raw text output
    raw_summary = response.choices[0].message.content.strip()    
    # Split the text into lines and clean up whitespace
    lines = raw_summary.split('\n')
    lines = [line.strip() for line in lines if line.strip()]    
    # Join the lines back together with actual line breaks
    formatted_summary = '\n'.join(lines)    
    return formatted_summary

# Generate and print the summary
summary = generate_summary(combined_text)
print(summary)

2026-10-05 11:09:55,589 INFO HTTP Request: POST http://localhost:11434/v1/chat/completions "HTTP/1.1 200 OK"


**Key Take‑aways**
---
### 1. NDRG1 and Oral Cancer (OSCC)
- **What was studied?**
Researchers looked at a protein called **NDRG1** in 100 patients with oral squamous cell carcinoma (OSCC) and followed them for at least 10 years.
- **Why is it important?**
The spread of cancer to nearby lymph nodes (regional metastasis) is the biggest factor that determines how well a patient does. Finding a marker that predicts this spread could help doctors decide who needs more aggressive treatment.
- **Main findings**
- **Low NDRG1 levels** were found in patients whose cancer had already spread to lymph nodes.
- Low NDRG1 also correlated with higher activity of enzymes (MMP‑2, ‑9, ‑10) that break down tissue and with a protein (BCL2) that helps cancer cells survive.
- Patients with low NDRG1 were more likely to have cancer come back or spread again.
- Even after accounting for other factors, NDRG1 remained an **independent predictor** of poor outcome.
- **What does it mean biologically?**
In lab ex

### Step 3: use a vector-powered knowledge graph to test data retrieval

Create a column to store URIs for each article

In [81]:
# Function to create a valid URI
def create_valid_uri(base_uri, text):
    if pd.isna(text):
        return None
    # Encode text to be used in URI
    sanitized_text = urllib.parse.quote(text.strip().replace(' ', '_').replace('"', '').replace('<', '').replace('>', '').replace("'", "_"))
    return URIRef(f"{base_uri}/{sanitized_text}")

# Add a new column to the DataFrame for the article URIs
df['Article_URI'] = df['Title'].apply(lambda title: create_valid_uri("http://example.org/article", title))


Test that the URI works - we are able to find all mesh terms associated with a given uri

In [ ]:
from rdflib import Graph, Namespace, URIRef

# Assuming your RDF graph (g) is already loaded
# Define namespaces
schema = Namespace('http://schema.org/')
ex = Namespace('http://example.org/')
rdfs = Namespace('http://www.w3.org/2000/01/rdf-schema#')

def get_mesh_terms_for_article(graph, article_uri):
    # Define the SPARQL query using the article URI
    query = f"""
    PREFIX schema: <http://schema.org/>
    PREFIX ex: <http://example.org/>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

    SELECT ?meshTermLabel
    WHERE {{
      <{article_uri}> a ex:Article ;
               schema:about ?meshTerm .
      ?meshTerm rdfs:label ?meshTermLabel .
    }}
    """
    
    # Execute the query
    results = graph.query(query)    
    # Extract the MeSH terms from the results
    mesh_terms = [str(row['meshTermLabel']) for row in results]    
    return mesh_terms

# Example usage with the provided URI
article_uri = "http://example.org/article/Expression_of_p53_and_coexistence_of_HPV_in_premalignant_lesions_and_in_cervical_cancer."
mesh_terms_example = get_mesh_terms_for_article(g, article_uri)
# Output the results
print(f"MeSH terms associated with the article '{article_uri}':")
for term in mesh_terms_example:
    print(term)

MeSH terms associated with the article 'http://example.org/article/Expression_of_p53_and_coexistence_of_HPV_in_premalignant_lesions_and_in_cervical_cancer.':
DNA Probes
HPV'
Viral'
'DNA
'Female'
'Humans'
'Immunohistochemistry'
'Papillomaviridae'
'Tumor Suppressor Protein p53'
'Uterine Cervical Dysplasia'
'Uterine Cervical Neoplasms


Create the schema for new embedding which includes mesh tags and URI

In [83]:
from weaviate.classes.config import Configure, Property, DataType

In [84]:
# Instance is capped at 1 collection; Articles from the section above must go first.
client.collections.delete("Articles")
client.collections.delete("Articles_with_abstracts_and_URIs")
client.collections.create(
    name="Articles_with_abstracts_and_URIs",
    properties=[
        Property(name="title", data_type=DataType.TEXT),
        Property(name="abstractText", data_type=DataType.TEXT),
        Property(name="meshMajor", data_type=DataType.TEXT),
        Property(name="article_URI", data_type=DataType.TEXT),
    ],
    vector_config=Configure.Vectors.text2vec_ollama(api_endpoint=OLLAMA_URL, model="nomic-embed-text"),
    generative_config=Configure.Generative.ollama(api_endpoint=OLLAMA_URL, model="gpt-oss:20b"),
)

2026-10-05 11:10:42,035 INFO HTTP Request: DELETE https://xzztqonlqni1m1kjnjovg.c0.us-east-1.aws.weaviate.cloud/v1/schema/Articles "HTTP/1.1 200 OK"
2026-10-05 11:10:42,233 INFO HTTP Request: DELETE https://xzztqonlqni1m1kjnjovg.c0.us-east-1.aws.weaviate.cloud/v1/schema/Articles_with_abstracts_and_URIs "HTTP/1.1 200 OK"
2026-10-05 11:10:42,474 INFO HTTP Request: POST https://xzztqonlqni1m1kjnjovg.c0.us-east-1.aws.weaviate.cloud/v1/schema "HTTP/1.1 200 OK"


In [ ]:
import logging
import numpy as np
from weaviate.util import generate_uuid5

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
# Replace infinity values with NaN and then fill NaN values
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.fillna('', inplace=True)
# Convert columns to string type
df['Title'] = df['Title'].astype(str)
df['abstractText'] = df['abstractText'].astype(str)
df['meshMajor'] = df['meshMajor'].astype(str)
df['Article_URI'] = df['Article_URI'].astype(str)
# Log the data types
logging.info(f"Title column type: {df['Title'].dtype}")
logging.info(f"abstractText column type: {df['abstractText'].dtype}")
logging.info(f"meshMajor column type: {df['meshMajor'].dtype}")
logging.info(f"Article_URI column type: {df['Article_URI'].dtype}")
articles = client.collections.get("Articles_with_abstracts_and_URIs")
with articles.batch.fixed_size(batch_size=50) as batch:
    for index, row in df.iterrows():
        try:
            question_object = {
                "title": row.Title,
                "abstractText": row.abstractText,
                "meshMajor": row.meshMajor,
                "article_URI": row.Article_URI,
            }
            batch.add_object(
                properties=question_object,
                uuid=generate_uuid5(question_object)
            )
        except Exception as e:
            logging.error(f"Error processing row {index}: {e}")

2026-10-05 11:10:53,781 INFO Title column type: str
2026-10-05 11:10:53,782 INFO abstractText column type: str
2026-10-05 11:10:53,782 INFO meshMajor column type: str
2026-10-05 11:10:53,783 INFO Article_URI column type: str
2026-10-05 11:10:54,007 INFO HTTP Request: GET https://xzztqonlqni1m1kjnjovg.c0.us-east-1.aws.weaviate.cloud/v1/schema/Articles_with_abstracts_and_URIs "HTTP/1.1 200 OK"


### Semantic search with vectorized KG

In [ ]:
response = client.collections.get("Articles_with_abstracts_and_URIs").query.near_text(
    query="mouth neoplasms",
    limit=10,
)
articles = [o.properties for o in response.objects]
print(json.dumps(articles, indent=4))

[
    {
        "meshMajor": "['Adult', 'Biomarkers, Tumor', 'Carcinoma, Mucoepidermoid', 'Diagnosis, Differential', 'Eosinophilia', 'Female', 'Fibrosis', 'Humans', 'Immunohistochemistry', 'Lymphatic Metastasis', 'Male', 'Middle Aged', 'Parotid Neoplasms', 'Sclerosis', 'Submandibular Gland Neoplasms']",
        "abstractText": "We encountered two cases of low malignant mucoepidermoid carcinoma with scanty cellular atypism which originated in the parotid or submandibular gland and was characterized by marked fibrosis and eosinophilic infiltration within tumor tissue despite the predominance of the squamous component. Here we report these two cases and provide a review of the literature. We believe that clinically these two tumors with stromal fibrosis and eosinophilic infiltration have a low malignant potential, although histological examination revealed a scanty mucus-producing epithelial component. Therefore, we consider this type of tumor as a new subtype of mucoepidermoid carcinoma.

In [87]:
# Extract article URIs
article_uris = [article["article_URI"] for article in articles]

In [88]:
article_uris

['http://example.org/article/Sclerosing_mucoepidermoid_carcinoma_with_eosinophilia_of_the_salivary_glands.',
 'http://example.org/article/Chondrosarcomatous_cutaneous_metastasis._A_unique_manifestation_of_sarcomatoid_%28metaplastic%29_breast_carcinoma.',
 'http://example.org/article/NDRG1_deficiency_is_associated_with_regional_metastasis_in_oral_cancer_by_inducing_epithelial-mesenchymal_transition.',
 'http://example.org/article/GSTM1%2C_GSTT1%2C_GSTP1%2C_CYP1A1%2C_and_NAT1_polymorphisms%2C_tobacco_use%2C_and_the_risk_of_head_and_neck_cancer.',
 'http://example.org/article/Juvenile_granulosa_cell_tumor_of_the_testis.',
 'http://example.org/article/Angiosarcoma_associated_with_germ_cell_tumors.',
 'http://example.org/article/Multiple_persistent_keratoacanthomas.',
 'http://example.org/article/No_germline_FH_mutations_in_familial_breast_cancer_patients.',
 'http://example.org/article/Dysbiosis_of_the_Gut_Microbiome_is_associated_with_Tumor_Biomarkers_in_Lung_Cancer.',
 'http://example.or

In [ ]:
from rdflib import Graph, Namespace, URIRef, Literal
from rdflib.namespace import RDF, RDFS, XSD

# Define namespaces
schema = Namespace('http://schema.org/')
ex = Namespace('http://example.org/')
rdfs = Namespace('http://www.w3.org/2000/01/rdf-schema#')
xsd = Namespace('http://www.w3.org/2001/XMLSchema#')

def get_articles_after_date(graph, article_uris, date_cutoff):
    # Create a dictionary to store results for each URI
    results_dict = {}

    # Define the SPARQL query using a list of article URIs and a date filter
    uris_str = " ".join(f"<{uri}>" for uri in article_uris)
    query = f"""
    PREFIX schema: <http://schema.org/>
    PREFIX ex: <http://example.org/>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>

    SELECT ?article ?title ?datePublished
    WHERE {{
      VALUES ?article {{ {uris_str} }}
      
      ?article a ex:Article ;
               schema:name ?title ;
               schema:datePublished ?datePublished .
      
      FILTER (?datePublished > "{date_cutoff}"^^xsd:date)
    }}
    """
    
    # Execute the query
    results = graph.query(query)    
    # Extract the details for each article
    for row in results:
        article_uri = str(row['article'])
        results_dict[article_uri] = {
            'title': str(row['title']),
            'date_published': str(row['datePublished'])
        }    
    return results_dict

date_cutoff = "2023-01-01"
articles_after_date = get_articles_after_date(g, article_uris, date_cutoff)

# Output the results
for uri, details in articles_after_date.items():
    print(f"Article URI: {uri}")
    print(f"Title: {details['title']}")
    print(f"Date Published: {details['date_published']}")
    print()

Article URI: http://example.org/article/Sclerosing_mucoepidermoid_carcinoma_with_eosinophilia_of_the_salivary_glands.
Title: Sclerosing mucoepidermoid carcinoma with eosinophilia of the salivary glands.
Date Published: 2026-06-25

Article URI: http://example.org/article/GSTM1%2C_GSTT1%2C_GSTP1%2C_CYP1A1%2C_and_NAT1_polymorphisms%2C_tobacco_use%2C_and_the_risk_of_head_and_neck_cancer.
Title: GSTM1, GSTT1, GSTP1, CYP1A1, and NAT1 polymorphisms, tobacco use, and the risk of head and neck cancer.
Date Published: 2026-06-07

Article URI: http://example.org/article/Juvenile_granulosa_cell_tumor_of_the_testis.
Title: Juvenile granulosa cell tumor of the testis.
Date Published: 2023-06-25

Article URI: http://example.org/article/No_germline_FH_mutations_in_familial_breast_cancer_patients.
Title: No germline FH mutations in familial breast cancer patients.
Date Published: 2023-05-31



### Similarity search with vectorized KG

In [ ]:
from weaviate.classes.query import MetadataQuery
coll = client.collections.get("Articles_with_abstracts_and_URIs")
seed = coll.query.fetch_objects(limit=1).objects[0]
response = coll.query.near_object(
    near_object=seed.uuid,
    limit=50,
    return_metadata=MetadataQuery(distance=True),
)
# Drop the seed itself (distance 0) so only genuinely similar articles are shown
others = [o for o in response.objects if o.uuid != seed.uuid]
for o in others:
    print(round(o.metadata.distance, 4), o.properties)

0.199 {'meshMajor': "['Adult', 'Cold Temperature', 'Double-Blind Method', 'Humans', 'MSH Release-Inhibiting Hormone', 'Male', 'Morphine', 'Pain', 'Time Factors']", 'abstractText': 'Prolyl-leucyl-glycinamide (MIF-1) has been observed to inhibit the analgesic effect of morphine in a series of animal studies. In the present study, the naloxone-like properties of MIF-1 were assessed in human subjects. Eight men received a capsule containing 60 mg of MIF-1 or placebo followed one hour later by a 10 mg intramuscular injection of morphine in a double-blind, crossover design at two visits 4 weeks apart. Experimental pain was induced by the cold pressor test administered 45, 75, 120 and 180 min after the morphine. Each subject recorded severity of pain on a 100 mm line scale every 5 sec during the 120 sec his foot was immersed in the cold water tank and during the 60 seconds immediately following its removal. On a third visit, baseline values were measured in the absence of morphine, MIF-1 or p

In [91]:
# Assuming articles is the list of article properties returned above
article_uris = [URIRef(article["article_URI"]) for article in articles]

In [ ]:
from rdflib import URIRef

# Constructing the SPARQL query with a FILTER for the article URIs
query = """
PREFIX schema: <http://schema.org/>
PREFIX ex: <http://example.org/>

SELECT ?article ?title ?abstract ?datePublished ?access ?meshTerm
WHERE {
  ?article a ex:Article ;
           schema:name ?title ;
           schema:description ?abstract ;
           schema:datePublished ?datePublished ;
           ex:access ?access ;
           schema:about ?meshTerm .

  ?meshTerm a ex:MeSHTerm .

  # Filter to include only articles from the list of URIs
  FILTER (?article IN (%s))
}
"""
# Convert the list of URIRefs into a string suitable for SPARQL
article_uris_string = ", ".join([f"<{str(uri)}>" for uri in article_uris])
# Insert the article URIs into the query
query = query % article_uris_string
# Dictionary to store articles and their associated MeSH terms
article_data = {}
# Run the query for each MeSH term
for mesh_term in mesh_terms:
    results = g.query(query, initBindings={'meshTerm': mesh_term})
    # Process results
    for row in results:
        article_uri = row['article']
        if article_uri not in article_data:
            article_data[article_uri] = {
                'title': row['title'],
                'abstract': row['abstract'],
                'datePublished': row['datePublished'],
                'access': row['access'],
                'meshTerms': set()
            }
        # Add the MeSH term to the set for this article
        article_data[article_uri]['meshTerms'].add(str(row['meshTerm']))
# Rank articles by the number of matching MeSH terms
ranked_articles = sorted(
    article_data.items(),
    key=lambda item: len(item[1]['meshTerms']),
    reverse=True
)
# Output results
for article_uri, data in ranked_articles:
    print(f"Title: {data['title']}")
    print(f"Abstract: {data['abstract']}")
    print("MeSH Terms:")
    for mesh_term in data['meshTerms']:
        print(f"  - {mesh_term}")
    print()

Title: NDRG1 deficiency is associated with regional metastasis in oral cancer by inducing epithelial-mesenchymal transition.
Abstract: Regional metastasis is the single most important prognostic factor in oral squamous cell carcinoma (OSCC). Abnormal expression of N-myc downstream-regulated genes (NDRGs) has been identified to occur in several tumor types and to predict poor prognosis. In OSCC, the clinical significance of deregulated NDRG expression has not been fully established. In this study, NDRG1 relevance was assessed at gene and protein levels in 100 OSCC patients followed up by at least 10 years. Survival outcome was analyzed using a multivariable analysis. Tumor progression and metastasis was investigated in preclinical model using oral cancer cell lines (HSC3 and SCC25) treated with epidermal growth factor (EGF) and orthotopic mouse model of metastatic murine OSCC (AT84). We identified NDRG1 expression levels to be significantly lower in patients with metastatic tumors compa

In [93]:
# List to store the URIs of the ranked articles
ranked_article_uris = [URIRef(article_uri) for article_uri, data in ranked_articles]

In [ ]:
from rdflib import Graph, Namespace, URIRef, Literal
from rdflib.namespace import RDF, RDFS, XSD, SKOS

# Assuming your RDF graph (g) is already loaded
# Define namespaces
schema = Namespace('http://schema.org/')
ex = Namespace('http://example.org/')
rdfs = Namespace('http://www.w3.org/2000/01/rdf-schema#')

def filter_articles_by_access(graph, article_uris, access_values):
    # Construct the SPARQL query with a dynamic VALUES clause
    uris_str = " ".join(f"<{uri}>" for uri in article_uris)
    query = f"""
    PREFIX schema: <http://schema.org/>
    PREFIX ex: <http://example.org/>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

    SELECT ?article ?title ?abstract ?datePublished ?access ?meshTermLabel
    WHERE {{
      VALUES ?article {{ {uris_str} }}
      
      ?article a ex:Article ;
               schema:name ?title ;
               schema:description ?abstract ;
               schema:datePublished ?datePublished ;
               ex:access ?access ;
               schema:about ?meshTerm .
      ?meshTerm rdfs:label ?meshTermLabel .
      
      FILTER (?access IN ({", ".join(map(str, access_values))}))
    }}
    """
    
    # Execute the query
    results = graph.query(query)    
    # Extract the details for each article
    results_dict = {}
    for row in results:
        article_uri = str(row['article'])
        if article_uri not in results_dict:
            results_dict[article_uri] = {
                'title': str(row['title']),
                'abstract': str(row['abstract']),
                'date_published': str(row['datePublished']),
                'access': str(row['access']),
                'mesh_terms': []
            }
        results_dict[article_uri]['mesh_terms'].append(str(row['meshTermLabel']))    
    return results_dict

access_values = [3,5,7]
filtered_articles = filter_articles_by_access(g, ranked_article_uris, access_values)
# Output the results
for uri, details in filtered_articles.items():
    print(f"Article URI: {uri}")
    print(f"Title: {details['title']}")
    print(f"Abstract: {details['abstract']}")
    print(f"Date Published: {details['date_published']}")
    print(f"Access: {details['access']}")
    print()

Article URI: http://example.org/article/NDRG1_deficiency_is_associated_with_regional_metastasis_in_oral_cancer_by_inducing_epithelial-mesenchymal_transition.
Title: NDRG1 deficiency is associated with regional metastasis in oral cancer by inducing epithelial-mesenchymal transition.
Abstract: Regional metastasis is the single most important prognostic factor in oral squamous cell carcinoma (OSCC). Abnormal expression of N-myc downstream-regulated genes (NDRGs) has been identified to occur in several tumor types and to predict poor prognosis. In OSCC, the clinical significance of deregulated NDRG expression has not been fully established. In this study, NDRG1 relevance was assessed at gene and protein levels in 100 OSCC patients followed up by at least 10 years. Survival outcome was analyzed using a multivariable analysis. Tumor progression and metastasis was investigated in preclinical model using oral cancer cell lines (HSC3 and SCC25) treated with epidermal growth factor (EGF) and ort

### RAG with a vectorized KG

In [ ]:
response = client.collections.get("Articles_with_abstracts_and_URIs").query.near_text(query="therapies for mouth neoplasms", limit=3)
context = "\n\n".join(o.properties["title"] + ": " + o.properties["abstractText"] for o in response.objects)
from openai import OpenAI
llm = OpenAI(base_url=os.getenv("OPENAI_API_BASE"), api_key="ollama")  # local generation, avoids the tunnel timeout
PROMPT = "Summarize the key information here in bullet points. Make it understandable to someone without a medical degree.\n\n" + context
out = llm.chat.completions.create(
    model="gpt-oss:20b",
    messages=[{"role": "user", "content": PROMPT}],
    max_tokens=2000,
)
print(out.choices[0].message.content)

2026-10-05 11:13:23,904 INFO HTTP Request: POST http://localhost:11434/v1/chat/completions "HTTP/1.1 200 OK"


**Key Take‑aways**

---

### 1. NDRG1 and spread of oral cancer
- **What was studied?**  
  Researchers looked at a protein called **NDRG1** in 100 patients with oral squamous cell carcinoma (OSCC) who were followed for at least 10 years.

- **Why it matters**  
  The single most important factor predicting worse outcomes in OSCC is whether the cancer has spread to nearby lymph nodes (regional metastasis). The study aimed to see if lower levels of NDRG1 are linked to that spread.

- **Main findings**  
  - Patients whose tumors had **low NDRG1 levels** were far more likely to have cancer that had already metastasized.  
  - Low NDRG1 was also correlated with higher activity of certain enzymes that break down tissue (MMP‑2, -9, -10) and a protein that helps cancer cells survive (BCL2).  
  - In statistical models that control for other factors, **low NDRG1 independently predicted poorer survival**.  
  - In cell‑culture experiments, exposing OSCC cells to epidermal growth factor (EGF) r

In [ ]:
# Extract article URIs
article_uris = [article["article_URI"] for article in articles]
# Function to filter the articles for only the given URIs
def filter_articles_by_uri(articles, article_uris):
    filtered_articles = []    
    for article in articles:
        if article['article_URI'] in article_uris:
            filtered_articles.append(article)    
    return filtered_articles
# Filter the articles
filtered_articles = filter_articles_by_uri(articles, article_uris)
# Output the filtered articles
print("Filtered articles:")
for article in filtered_articles:
    print(f"Title: {article['title']}")
    print(f"URI: {article['article_URI']}")
    print(f"Abstract: {article['abstractText']}")
    print(f"MeshMajor: {article['meshMajor']}")
    print("---")

Filtered articles:
Title: Sclerosing mucoepidermoid carcinoma with eosinophilia of the salivary glands.
URI: http://example.org/article/Sclerosing_mucoepidermoid_carcinoma_with_eosinophilia_of_the_salivary_glands.
Abstract: We encountered two cases of low malignant mucoepidermoid carcinoma with scanty cellular atypism which originated in the parotid or submandibular gland and was characterized by marked fibrosis and eosinophilic infiltration within tumor tissue despite the predominance of the squamous component. Here we report these two cases and provide a review of the literature. We believe that clinically these two tumors with stromal fibrosis and eosinophilic infiltration have a low malignant potential, although histological examination revealed a scanty mucus-producing epithelial component. Therefore, we consider this type of tumor as a new subtype of mucoepidermoid carcinoma. A low-malignant mucoepidermoid carcinoma with stromal fibrosis and eosinophilic infiltration, as describe

## RAG on vectorized knowledge graph with filters

In [ ]:
response = client.collections.get("Articles_with_abstracts_and_URIs").query.near_text(
    query="therapies for mouth neoplasms",
    limit=20,
)
articles = [o.properties for o in response.objects]
# Assuming articles is the list of article properties returned above
article_uris = [URIRef(article["article_URI"]) for article in articles]

In [ ]:
from rdflib import URIRef
# Constructing the SPARQL query with a FILTER for the article URIs
query = """
PREFIX schema: <http://schema.org/>
PREFIX ex: <http://example.org/>

SELECT ?article ?title ?abstract ?datePublished ?access ?meshTerm
WHERE {
  ?article a ex:Article ;
           schema:name ?title ;
           schema:description ?abstract ;
           schema:datePublished ?datePublished ;
           ex:access ?access ;
           schema:about ?meshTerm .

  ?meshTerm a ex:MeSHTerm .

  # Filter to include only articles from the list of URIs
  FILTER (?article IN (%s))
}
"""
# Convert the list of URIRefs into a string suitable for SPARQL
article_uris_string = ", ".join([f"<{str(uri)}>" for uri in article_uris])
# Insert the article URIs into the query
query = query % article_uris_string
# Dictionary to store articles and their associated MeSH terms
article_data = {}
# Run the query for each MeSH term
for mesh_term in mesh_terms:
    results = g.query(query, initBindings={'meshTerm': mesh_term})
    # Process results
    for row in results:
        article_uri = row['article']
        if article_uri not in article_data:
            article_data[article_uri] = {
                'title': row['title'],
                'abstract': row['abstract'],
                'datePublished': row['datePublished'],
                'access': row['access'],
                'meshTerms': set()
            }
        # Add the MeSH term to the set for this article
        article_data[article_uri]['meshTerms'].add(str(row['meshTerm']))
# Rank articles by the number of matching MeSH terms
ranked_articles = sorted(
    article_data.items(),
    key=lambda item: len(item[1]['meshTerms']),
    reverse=True
)
# Output results
for article_uri, data in ranked_articles:
    print(f"Title: {data['title']}")
    print(f"Abstract: {data['abstract']}")
    print("MeSH Terms:")
    for mesh_term in data['meshTerms']:
        print(f"  - {mesh_term}")
    print()

Title: NDRG1 deficiency is associated with regional metastasis in oral cancer by inducing epithelial-mesenchymal transition.
Abstract: Regional metastasis is the single most important prognostic factor in oral squamous cell carcinoma (OSCC). Abnormal expression of N-myc downstream-regulated genes (NDRGs) has been identified to occur in several tumor types and to predict poor prognosis. In OSCC, the clinical significance of deregulated NDRG expression has not been fully established. In this study, NDRG1 relevance was assessed at gene and protein levels in 100 OSCC patients followed up by at least 10 years. Survival outcome was analyzed using a multivariable analysis. Tumor progression and metastasis was investigated in preclinical model using oral cancer cell lines (HSC3 and SCC25) treated with epidermal growth factor (EGF) and orthotopic mouse model of metastatic murine OSCC (AT84). We identified NDRG1 expression levels to be significantly lower in patients with metastatic tumors compa

In [ ]:
# Function to combine titles and abstracts
def combine_abstracts(ranked_articles):
    combined_text = "".join(
        [f"Title: {data['title']} Abstract: {data['abstract']}" for article_uri, data in ranked_articles]
    )
    return combined_text
# Combine abstracts from the top 3 articles
combined_text = combine_abstracts(ranked_articles)
print(combined_text)

Title: NDRG1 deficiency is associated with regional metastasis in oral cancer by inducing epithelial-mesenchymal transition. Abstract: Regional metastasis is the single most important prognostic factor in oral squamous cell carcinoma (OSCC). Abnormal expression of N-myc downstream-regulated genes (NDRGs) has been identified to occur in several tumor types and to predict poor prognosis. In OSCC, the clinical significance of deregulated NDRG expression has not been fully established. In this study, NDRG1 relevance was assessed at gene and protein levels in 100 OSCC patients followed up by at least 10 years. Survival outcome was analyzed using a multivariable analysis. Tumor progression and metastasis was investigated in preclinical model using oral cancer cell lines (HSC3 and SCC25) treated with epidermal growth factor (EGF) and orthotopic mouse model of metastatic murine OSCC (AT84). We identified NDRG1 expression levels to be significantly lower in patients with metastatic tumors compa

In [ ]:
# Function to combine titles and abstracts into one chunk of text
def combine_abstracts(ranked_articles):
    combined_text = " ".join(
        [f"Title: {data['title']} Abstract: {data['abstract']}" for _, data in ranked_articles]
    )
    return combined_text
# Combine abstracts from the filtered articles
combined_text = combine_abstracts(ranked_articles)
print(combined_text)

Title: NDRG1 deficiency is associated with regional metastasis in oral cancer by inducing epithelial-mesenchymal transition. Abstract: Regional metastasis is the single most important prognostic factor in oral squamous cell carcinoma (OSCC). Abnormal expression of N-myc downstream-regulated genes (NDRGs) has been identified to occur in several tumor types and to predict poor prognosis. In OSCC, the clinical significance of deregulated NDRG expression has not been fully established. In this study, NDRG1 relevance was assessed at gene and protein levels in 100 OSCC patients followed up by at least 10 years. Survival outcome was analyzed using a multivariable analysis. Tumor progression and metastasis was investigated in preclinical model using oral cancer cell lines (HSC3 and SCC25) treated with epidermal growth factor (EGF) and orthotopic mouse model of metastatic murine OSCC (AT84). We identified NDRG1 expression levels to be significantly lower in patients with metastatic tumors compa

In [ ]:
# Generate and print the summary
summary = generate_summary(combined_text)
print(summary)

2026-10-05 11:14:25,371 INFO HTTP Request: POST http://localhost:11434/v1/chat/completions "HTTP/1.1 200 OK"


**Key Take‑aways – NDRG1 and Oral Cancer**
- **What was studied?**
Researchers looked at a protein called **NDRG1** in 100 patients with oral squamous cell carcinoma (OSCC) and followed them for at least 10 years.
- **Why is it important?**
The spread of cancer to nearby lymph nodes (regional metastasis) is the biggest factor that determines a patient’s outlook.
- **Main findings**
- Patients whose tumors had **low NDRG1 levels** were far more likely to have cancer that had spread to lymph nodes.
- Low NDRG1 also correlated with higher levels of enzymes (MMP‑2, ‑9, ‑10) that break down tissue and a protein (BCL2) that helps cancer cells survive.
- In lab experiments, reducing NDRG1 made oral cancer cells behave more like “invasive” cells (they changed shape and moved more easily – a process called epithelial‑mesenchymal transition or EMT).
- In a mouse model, the same pattern was seen: low NDRG1 → more spread of cancer.
- **What does it mean for patients?**
- **Low NDRG1** can serve as